---
TOP 10 FILES FOR THE AIC Competition -- Deep Walkthrough

---
FILE 1: `policy.py` - The Contract You Must Fulfill

PATH: `aic_model/aic_model/policy.py` (146 lines)
IMPORTANCE: This is THE file. Everything you build inherits from this.


KEY LINES EXPLAINED
    Lines 36-68 - THE THREE CALLBACKS (your API to the world)

    `GetObservationCallback = Callable[[], Observation]     # Line 36`
    This is a function type. When you call `get_observation()` inside your
    policy, it returns the latest `Observation` message -- all 3 cameras, joint
    states, wrench data, controller state. Delivered at 20 Hz. 

```Python
class MoveRobotCallback(Protocol):
    def __call__(
        self,
        motion_update: MotionUpdate = None,                 # Cartesian control
        joint_motion_update: JointMotionUpdate = None,      # Joint control
    ) -> None: ...
```
   This is how you command the robot. You pass EITHER a `MotionUpdate` 
   (Cartesian pose/velocity) OR a `JointMotionUpdate` (joint angles). Never
   both at once. The system auto-switches control modes for you.


```Python
SendFeedbackCallback = Callable[[str], None]                # Line 68
```
   Sends a progress string back to the engine. Use it for logging/debugging.


---
LINES 71-88 -- The Policy Base Class
```Python
class Policy(ABC):
    def __init__(self, parent_node):
        self._parent_node = parent_node         # Stores the ROS node
```
   `parent_node` is the `AicModel` lifecycle node. Through it you access the
   ROS clock, logger, and TF buffer. This is critical -- 
   `self._parent_node._tf_buffer` is how CheatCode looks up transforms.

```Python
    def time_now(self):
        return self.get_clock().now()

    def sleep_for(self, duration_sec: float) -> None:
        self.get_clock().sleep_for(Duration(seconds=duration_sec))
```
   These use SIM TIME, not wall time. The sim clock can run faster or slower
   than real time. Always use these instead of `time.sleep()` for anything
   physics-coupled. 


---
LINES 90-134 -- `set_pose_target()` -- The Convenience Helper
```Python
    def set_pose_target(
        self, 
        move_robot: MoveRobotCallback, 
        pose: Pose, 
        frame_id, 
        str = "base_link"
    ) -> None:
```
   This wraps up a `MotionUpdate` message with sensible defaults. The key 
   parameters it sets:

```Python
        motion_update = MotionUpdate(
            target_stiffness=np.diag([90.0, 90.0, 90.0, 50.0, 50.0, 50.0]).flatten(),  # Line 120
            target_damping=np.diag([50.0, 50.0, 50.0, 20.0, 20.0, 20.0]).flatten(),    # Line 121
        )
```
   - Stiffness [90, 90, 90, 50, 50, 50]: How strongly the arm tracks the target
     pose. First 3 = translational (N/m), last 3 = rotational (Nm/rad). Higher
     = stiffer tracking, more jerk. 
   - Damping [50, 50, 50, 20, 20, 20]: Velocity-proportional resistance. 
     Higher = smoother but slower. 


```Python
        wrench_feedback_gains_at_tip=[0.5, 0.5, 0.5, 0.0, 0.0, 0.0], # Line 126
```
   Force feedback gains. [0.5, 0.5, 0.5] means 50% force feedback on translation
   (the arm yields to external forces). [0, 0, 0] on rotation means no 
   rotational compliance. Range is [0, 0.95].


```Python
              trajectory_generation_mode=TrajectoryGenerationMode(
                  mode=TrajectoryGenerationMode.MODE_POSITION,               # Line 128
              ),
```
   `MODE_POSITION` means the pose field is used. `MODE_VELOCITY` means the
   velocity field is used instead. 


---
LINES 136-145 -- The Abstract Method You Implement
```Python
        @abstractmethod
        def insert_cable(
            self,
            task: Task,                                 # What to insert where
            get_observation: GetObservationCallback,    # See the world
            move_robot: MoveRobotCallback,              # Command the arm
            send_feedback: SendFeedbackCallback,        # Report progress
        )
```
   



---


   In Python, a callback is a function that you pass as an argument to another
   function. The receiving function can then callback at a later point, often
   as a response to an event or after completing a task. 



---
ROS CLOCK
   The ROS clock is the centralised timekeeping mechanism for the entire robot
   system, and in the AIC competition, it is accessed via `self.time_now()` or
   `self.get_clock()`. Unlike standard system time, the ROS clock follows
   SIMULATION TIME when running in Gazebo or Isaac Sim, meaning it can be paused
   , slowed down, or sped up without breaking the logic of your policy. You must
   use this clock for all timing operations--such as the `self.sleep_for()` 
   method--to ensure that your robot's movements remain synchronised with the 
   physics engine. Using a standard Python `time.sleep()` would cause the robot
   to "desync" because the real-world clock keeps ticking even if the simulation
   pauses to calculate a complex collision.


ROS LOGGER
   The ROS logger is the standard diagnostic tool for tracking your policy's
   internal state and errors, accessed through `self.get_logger()`. It allows
   you to output messages at different severity levels--such as `info`, `warn`,
   or `error`--which are then captured by the ROS 2 framework and can be viewed
   in the terminal or saved to log files for post-trail analysis. During the
   cable insertion task, you can use `send_feedback()` to pass progress strings
   back to the engine, but the logger is your primary tool for deep technical
   debugging, such as printing out current force-torque values or the status of
   your vision model. This is essential for understanding why a trail railed 
   once the robot is running autonomously in a Docker container where you cannot 
   see the live 3D viewport. 


TF BUFFER
   The TF (Transform) buffer is a dedicated data structure that tracks the 3D
   coordinate relationships between every part of the robot and its environment
   over time. In your policy, you access this via `self._parent_node._tf_buffer`
   , which allows you to "look up" where the plug tip is located relative to the
   SFP port. The buffer is constantly updated by a `TransformListener` that 
   listens to the `/tf` and `/tf_static` topics. While the `CheatCode.py`
   reference uses this buffer to find the exact "ground truth" location of ports
   , in the actual evaluation, these environmental frames will be hidden, and
   you will primarily use the buffer to track the robot's own internal kinematic
   chain (e.g., from `base_link` to the `hand_e_gripper`), 





---
   ... difference between these two updates defines how you "speak" to the UR5e
   robot's controller.


1. MotionUpdate (Cartesian Control)
   This moves the robot's END-EFFECTOR (TCP) in 3D space. You are telling the
   robot where the gripper should be relative to the world or the robot's base. 

   * TARGET: Uses `geometry_msgs/Pose` (XYZ coordinates and a Quaternion for 
     rotation).
   * PHYSICS: Employs CARTESIAN IMPEDANCE CONTROL. You define a 6x6 stiffness
     matrix and a 6x6 damping matrix to determine how "soft" or "hard" the arm
     tracks that point.
   * BEST USE CASE: The INSERTION PHASE. You need the plug to stay perfectly
     aligned with the port's XYZ coordinates in the world. 


2. JointMotionUpdate (Joint-Space Control)
   This moves the robot's INDIVIDUAL MOTORS. You are telling the robot what
   angle each of its 6 joints should be. 

   * TARGET: Uses `trajectory_msgs/JointTrajectoryPoint` (a list of 6 radian
     values for the UR5e joints).
   * PHYSICS: Employs JOINT-SPACE IMPEDANCE CONTROL. Instead of a 6x6 matrix, 
     you define simple stiffness and damping values for each specific joint. 
   * BEST USE CASE: The APPROACH or HOME PHASE. When moving the arm from its
     starting position to near the task board, joint control is often smoother
     and avoids mathematical "singularities".  


KEY COMPARISON
   Feature // `MotionUpdate` //  `JointMotionUpdate`
   INPUT TYPE // Pose (Position + Rotation) // Joint Angles (Radians)
   LOGIC // "Put the SFP plug at this XYZ" // "Rotate Joint 3 by 45 degrees"
   IMPEDANCE // 6x6 matrices (row-major) // 6 individual joint values
   AUTO-SWITCHING // Handled by `aic_model.py` // Handled by `aic_model.py`

   
   The `aic_model.py` node is smart enough to see which message you sent and
   automatically call the `ChangeTargetMode` service to switch the controller
   for you.




-- An end-effector, often called End-of-Arm Tooling (EOAT), is the device 
   attached to the end of a robotic arm that allows it to interact with its 
   environment, acting as the robot's "hand". These tools enable robots to 
   perform specific tasks such as gripping, welding, cutting, or sensing. Common
   types include mechanical grippers, vacuum suction cups, and specialised tools
   for manufacturing. 



-- Tool Center Point (TCP) in robotics is the precise, user-defined coordinate
   point at the tip of robot's tool (e.g., welder, gripper, or nozzle) that the
   controller uses for motion planning and path execution. It enables accurate
   navigation by shifting the coordinate system from the robot base to the tool
   tip, allowing for rotation around that point rather than the robot flange.
   Correct TCP calibration is essential for precision, repeatability, and 
   preventing collisions. 


-- Cartesian impedance control is a robot motion strategy that regulates the
   relationship between end-effector motion (position/orientation) and external
   forces in Cartesian space, allowing for compliant, human-safe interaction.
   It simulates a virtual spring-damper system, enabling the robot to adjust to
   unknown environments and, if necessary, be guided by hand. 


---

  For tomorrow (Isaac Sim integration):
  1. Get Isaac Sim running with the UR5e model (you already know how to spawn articulations from
  Tutorial 3)
  2. Port the AIC task board description into Isaac Lab's USD format
  3. Set up the ROS 2 bridge between Isaac Sim and the AIC controller stack
  4. Test with a simple policy (like WaveArm) to verify the full pipeline works


  The AIC repo uses Gazebo as its default simulator, but the architecture is simulator-agnostic through
   ROS 2. The key is getting Isaac Sim to publish the same topics (/observation, camera images, joint
  states, F/T data) and subscribe to the same commands (MotionUpdate, JointMotionUpdate) that the AIC
  controller expects.


---    
3. WHAT DOES `write_root_*` DO?
    You are correct--it writes to the SIMULATION BUFFER.

    In high-peformance simulation, you don't move objects by talking to the "UI"
    or the Scene Tree. That is too slow. Instead:

    1. `write_root_pose_to_sim`: This teleports the obejcts to the new XYZ
       coordinates and rotations you just calculated.    
    2. `write_root_velocity_to_sim`: This sets their speed. In this script, it
       usually      

                    https://gemini.google.com/app/d4324975ae77fb67

hi. thanks. can you now on the other hand explain to me how does run_articulation.py work... and how it differs from run_rigid_object.py? thanks!

                    https://gemini.google.com/app/d4324975ae77fb67

finally! please similarly do such too for run_deformable_object.py

                    https://gemini.google.com/app/d4324975ae77fb67